In [1]:
from src.vanna_connector import initialize_vanna
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL,\
        OPENAI_API_URL, DENSE_EMBEDDING_MODEL_PATH, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE
from tqdm import tqdm
import pandas as pd
import os

/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Initialize Vanna client

In [2]:
# initialize configs for the database, vector store and chat model

PATH_TO_DB = os.path.join(PROCESSED_DATA_DIR, "bank_transaction_monitoring", "bank_transaction_monitoring_inline_short.sqlite.db")

# postgress example, for concrete params for different databases check VannaBase class methods connect_to_*
# postgres_config = {
#     "params": {
#         "host": "localhost",
#         "port": 5432,
#         "database": "bank_transaction_monitoring",
#         "user": "postgres",
#         "password": "postgres"
#     },
#     "type": "postgres"}


sqlite_config = {
    "params": {
        "url": str(PATH_TO_DB)
    },
    "type": "sqlite"}


qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [3]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2360.51it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


# Upload Data

## Upload DDL schemas with comments
Comments are needed to improve end-to-end quality by enriching context, explaining ambiguous info, describing fields which are in opaque like style, improving the quality of both search and SQL-generation

In [4]:
# load ddls from somewhere
DDL_PATH = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring/table_ddls_inline_short.csv")

ddls = pd.read_csv(DDL_PATH)['ddl'].tolist()

print(ddls[0])

CREATE TABLE btm_cst ( -- Клиенты банка.
    c01 INTEGER, -- Идентификатор клиента.
    c02 TEXT   , -- Имя клиента.
    c03 TEXT   , -- Адрес клиента.
    c04 TEXT   , -- Код штата клиента.
    c05 TEXT     -- Телефон клиента.
);


In [5]:
# upload them
for ddl_script in tqdm(ddls, total=len(ddls)):
    vanna_client.add_ddl(ddl_script)

100%|██████████| 8/8 [00:00<00:00, 10.67it/s]


## Uploading documentation
You can and must also add some additional info, documentation, rules description, etc. That has a relation to the database and can help in it's understanding

In [6]:
# load documentation from somewhere

doc_1 = """База bank_transaction_monitoring описывает клиентов банка, их счета, связи между счетами, операции, уведомления и процентные ставки.
Клиенты хранятся в btm_cst, где c01 это идентификатор клиента, c02 имя, c04 региональный код клиента.
Основные сведения о счетах находятся в btm_accd и btm_accs: номер счета, тип продукта, баланс, статус и роль клиента.
Поле a06 или s06 показывает роль клиента по счету: P означает основной владелец, S означает вторичная связь.
Операции по счетам хранятся в btm_trn, где t01 это номер счета, t02 сумма операции, t03 канал, t04 регион, t05 дата.
Положительные суммы операций означают поступления, отрицательные суммы означают списания или расходы.
Связи между продуктами отражены в btm_rel: r02 это связанный счет, r04 основной или родительский счет.
Тексты уведомлений для клиентов находятся в btm_msg и могут использоваться для задач коммуникации по событиям.
Процентные ставки по типам продуктов и периодам находятся в btm_rate.
Для соединения клиентов со счетами используйте btm_cst.c01 = btm_accd.a01 или btm_cst.c01 = btm_accs.s01; для соединения счетов с операциями используйте btm_accd.a02 = btm_trn.t01 или btm_accs.s02 = btm_trn.t01."""

docs = [doc_1]

In [7]:
# upload them
for document in tqdm(docs, total=len(docs)):
    vanna_client.add_documentation(document)

100%|██████████| 1/1 [00:00<00:00, 11.45it/s]


## Uploading database tables schema info
Note that this schema may not be provided by the chosen relational database (like sqlite), in this case use uploading ddl to do it manually. The example below shows how it can be done to the databases that provide such info

In [ ]:
# df_information_schema = vanna_client.run_sql("SELECT * FROM INFORMATION_SCHEMA.COLUMNS")

# # This will break up the information schema into bite-sized chunks that can be referenced by the LLM
# plan = vanna_client.get_training_plan_generic(df_information_schema)
# print(plan)
# vanna_client.train(plan=plan)

## Uploading SQL scripts with descriptions
Core functionality that is esential for the core functionality of searching similar SQL queries and generating new queries using stored examples. Requires pairs of SQL description and SQL script itself.

In [8]:
# load data from somewhere
QUERY_DESC_PATH = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring/query_descriptions/query_descriptions_inline_short_rewritten.csv")

query_desc_df = pd.read_csv(QUERY_DESC_PATH)

pairs = []

for index, row in query_desc_df.iterrows():
    description = row['query_business']
    uuid = row['uuid']
    if f'{uuid}.txt' in os.listdir(os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring",  "scripts")):
        with open(os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring/scripts", f'{uuid}.txt'), 'r') as f:
            sql_script = f.read()

        pairs.append((description, sql_script))

In [9]:
print(len(pairs))
print("description: ", pairs[0][0])
print("sql_script: ", pairs[0][1])

45
description:  Скрипт для поиска последних 10 операций по счетам, проведённых через канал ATM withdrawal, с выводом номера счета, суммы, штата и даты операции и сортировкой по дате операции по убыванию.
sql_script:  SELECT
  t01 AS account_number,
  t02 AS transaction_amount,
  t04 AS transaction_state,
  t05 AS transaction_date
FROM btm_trn
WHERE t03 = 'ATM withdrawal'
ORDER BY t05 DESC
LIMIT 10;


In [10]:
for description, sql_script in tqdm(pairs, total=len(pairs)):
    vanna_client.add_question_sql(question=description, sql=sql_script)

100%|██████████| 45/45 [00:01<00:00, 23.13it/s]
